# Visualiseur des videos one-cycle "choroide entiere"

Parcourt et visualise les sorties de `Astronauts/compute_one_cycle_whole_choroid.py` :
videos one-cycle repliees sur le pouls de la choroide entiere (SVD multiresolution +
combinaison optimisee, repris de `phase_shift_pixel_svd/conditions.csv`) via une phase
de Hilbert.

Chaque condition fournit trois fichiers, ecrits cote a cote sous
`SEGVAR_ROOT/<variante>/whole_choroid_one_cycle/<astro>/<moment>/<condition>/` :

| fichier | contenu |
|---|---|
| `one_cycle.mp4` | `n_cycle x n_bins` images : `n_cycle` cycles moyens successifs, `n_bins` phases chacun |
| `one_cycle_params.json` | parametres du repliement + verification de la reconstruction du pouls |
| `pixel_svd_diagnostics.npz` | ROI, composantes SVD, pouls, phase de Hilbert |

Le notebook enchaine : **inventaire** (toutes les conditions) -> **selection d'une
condition** -> planche de contact, animation, dynamique du cycle, pouls/phase, motif
spatial -> **survol de toutes les conditions**. Pour parcourir, il suffit de changer
`SELECT` puis de reexecuter les cellules a partir de la selection.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

from ocularrigidity.data.compression import read_gray

# --- Ou regarder (memes constantes que le script) --------------------------
SEGVAR_ROOT = Path("E:/NASA_Rigidity/SegmentationVariations")
MASK_VARIANT = "model1_scale_1.0"
OUTPUT_SUBDIR = "whole_choroid_one_cycle"
PULSE_SUBDIR = "phase_shift_pixel_svd"

OUT_ROOT = SEGVAR_ROOT / MASK_VARIANT / OUTPUT_SUBDIR
SUMMARY_CSV = SEGVAR_ROOT / "whole_choroid_one_cycle_summary.csv"
PULSE_CSV = SEGVAR_ROOT / MASK_VARIANT / PULSE_SUBDIR / "conditions.csv"

plt.rcParams["figure.dpi"] = 110
plt.rcParams["image.cmap"] = "gray"
print(OUT_ROOT, "->", "existe" if OUT_ROOT.is_dir() else "ABSENT (lancer le script d'abord)")

## 1. Inventaire

Une ligne par `one_cycle.mp4` present sur le disque, enrichie du resume
(`whole_choroid_one_cycle_summary.csv`) quand il existe. `rebuild_ok` dit si le pouls
reconstruit correspondait bien, aux tolerances pres, a celui enregistre dans
`conditions.csv` -- c'est la garantie que la video a bien ete repliee sur le pouls
"choroide entiere" deja publie, et non sur un pouls recalcule autrement.

In [ ]:
def scan_cases(out_root: Path) -> pd.DataFrame:
    rows = []
    for video in sorted(out_root.glob("*/*/*/one_cycle.mp4")):
        d = video.parent
        meta = {}
        params = d / "one_cycle_params.json"
        if params.exists():
            meta = json.loads(params.read_text(encoding="utf-8"))
        rows.append({
            "patient": d.parent.parent.name,
            "moment": d.parent.name,
            "condition": d.name,
            "cardiac_bpm": meta.get("cardiac_bpm", np.nan),
            "hr_visit": meta.get("hr_bpm_visit_data", np.nan),
            "n_cycle": meta.get("n_cycle", np.nan),
            "n_bins": meta.get("n_bins", np.nan),
            "fold": meta.get("fold_method", ""),
            "n_selected": len(meta.get("selected_indices", [])),
            "gap": meta.get("gap_fraction", np.nan),
            "notes": len(meta.get("notes", [])),
            "dir": d,
        })
    cases = pd.DataFrame(rows)
    if SUMMARY_CSV.exists() and not cases.empty:
        summary = pd.read_csv(SUMMARY_CSV, keep_default_na=False, na_values=["<NA>", ""])
        keep = [c for c in ("patient", "moment", "condition", "frac_in_band",
                            "peak_bpm", "corr_raw", "rebuild_ok", "confidence")
                if c in summary.columns]
        cases = cases.merge(summary[keep], on=["patient", "moment", "condition"], how="left")
    return cases


cases = scan_cases(OUT_ROOT)
print(f"{len(cases)} condition(s) avec une video one-cycle")
if PULSE_CSV.exists():
    n_ok = (pd.read_csv(PULSE_CSV, keep_default_na=False,
                        na_values=["<NA>", ""])["status"] == "ok").sum()
    print(f"({n_ok} pouls exploitables dans {PULSE_CSV.name} -> autant de videos attendues)")
cases

## 2. Selection d'une condition

`SELECT` accepte un indice de la table ci-dessus, ou n'importe quel bout de chemin
(`"210713001before"`, `"_OS1"`, ...) : la premiere condition qui correspond est prise.

In [ ]:
SELECT = 0          # indice dans `cases`, ou fragment de nom
CHUNK = "moyenne"   # "moyenne" (tous les cycles moyennes) ou un indice de cycle : 0, 1, 2...


def pick(cases: pd.DataFrame, select):
    if isinstance(select, (int, np.integer)):
        return cases.iloc[int(select)]
    key = str(select).lower()
    hit = cases[cases.apply(
        lambda r: key in f"{r['patient']}/{r['moment']}/{r['condition']}".lower(), axis=1)]
    if hit.empty:
        raise KeyError(f"aucune condition ne correspond a {select!r}")
    return hit.iloc[0]


def load_case(row):
    d = Path(row["dir"])
    meta = json.loads((d / "one_cycle_params.json").read_text(encoding="utf-8"))
    diag = dict(np.load(d / "pixel_svd_diagnostics.npz", allow_pickle=True))
    video = read_gray(d / "one_cycle.mp4").astype(np.float32)  # (n_cycle*n_bins, H, W)
    n_cycle, n_bins = int(meta["n_cycle"]), int(meta["n_bins"])
    cycles = video.reshape(n_cycle, n_bins, *video.shape[1:])
    # Un chunk sans assez de frames valides est ecrit tout noir (cf. NCycleReconstructor).
    filled = np.array([c.any() for c in cycles])
    return dict(row=row, dir=d, meta=meta, diag=diag, cycles=cycles,
                n_cycle=n_cycle, n_bins=n_bins, filled=filled)


case = load_case(pick(cases, SELECT))
c = case["cycles"]
print(f"{case['row']['patient']} / {case['row']['moment']} / {case['row']['condition']}")
print(f"  {case['n_cycle']} cycles x {case['n_bins']} phases, images {c.shape[2]}x{c.shape[3]} "
      f"({case['meta']['fold_method']})")
print(f"  cycles remplis : {np.flatnonzero(case['filled']).tolist()} "
      f"{'' if case['filled'].all() else '<- certains cycles ont ete sautes'}")

In [ ]:
m, d = case["meta"], case["diag"]
chk = m["rebuild_check"]
print(f"FC utilisee        : {m['cardiac_bpm']:.1f} BPM   (visit_data : {m['hr_bpm_visit_data']:.1f}, "
      f"source : {m['hr_source']})")
print(f"pouls repris       : {len(m['selected_indices'])} composantes SVD sur {m['k_svd']}, "
      f"{m['n_traces']} traces (blocs {m['block_sizes']})")
print(f"verification       : pic {chk['peak_bpm'][0]:.2f} vs {chk['peak_bpm'][1]:.2f} BPM · "
      f"en bande {chk['frac_in_band'][0]:.3f} vs {chk['frac_in_band'][1]:.3f} · "
      f"objectif {chk['objective'][0]:.5g} vs {chk['objective'][1]:.5g}"
      + ("  [OK]" if not chk["failed"] else f"  [ECART] {chk['failed']}"))
print(f"phase de Hilbert   : gap {m['gap_fraction']:.1%}, "
      f"passe-bande prealable : {m['bandpass_before_hilbert']}, confiance {m['confidence']}")
if m["notes"]:
    print("notes :")
    for n in m["notes"]:
        print("   -", n)

## 3. Planche de contact

Ligne du haut : le cycle moyen, phase par phase. Lignes suivantes : l'**ecart a la
moyenne du cycle** (meme echelle symetrique pour toutes les phases) -- c'est la que se
voit la pulsation, l'image brute etant dominee par l'anatomie statique.

In [ ]:
def chunk_view(case, chunk):
    # Cycle a afficher : moyenne des cycles remplis, ou l'un d'eux.
    cycles, filled = case["cycles"], case["filled"]
    if isinstance(chunk, str):
        return cycles[filled].mean(axis=0), "moyenne des cycles"
    return cycles[int(chunk)], f"cycle {int(chunk)}"


cycle, cycle_label = chunk_view(case, CHUNK)
delta = cycle - cycle.mean(axis=0, keepdims=True)
vmax = float(np.percentile(np.abs(delta), 99.5))
n_bins = case["n_bins"]

fig, axes = plt.subplots(2, n_bins, figsize=(1.5 * n_bins, 5.2))
for b in range(n_bins):
    axes[0, b].imshow(cycle[b], vmin=0, vmax=255)
    axes[0, b].set_title(f"{b / n_bins:.0%}", fontsize=8)
    axes[1, b].imshow(delta[b], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    for ax in (axes[0, b], axes[1, b]):
        ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel("image", fontsize=8)
axes[1, 0].set_ylabel(f"ecart\n(+/- {vmax:.1f} NG)", fontsize=8)
fig.suptitle(f"{case['row']['condition']} — {cycle_label}, une colonne par phase du cycle",
             fontsize=10)
fig.tight_layout()

## 4. Animation du cycle

Le cycle est joue en boucle (lecteur `to_jshtml`, autonome dans le notebook). A gauche
l'image, a droite l'ecart a la moyenne du cycle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
im0 = axes[0].imshow(cycle[0], vmin=0, vmax=255)
im1 = axes[1].imshow(delta[0], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("image"); axes[1].set_title("ecart a la moyenne")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
title = fig.suptitle("")
plt.close(fig)


def _frame(b):
    im0.set_data(cycle[b])
    im1.set_data(delta[b])
    title.set_text(f"{case['row']['condition']} — {cycle_label} — phase {b / n_bins:.0%}")
    return im0, im1, title


anim = animation.FuncAnimation(fig, _frame, frames=n_bins, interval=120, blit=False)
HTML(anim.to_jshtml(fps=8))

## 5. Dynamique du cycle

- **Kymographe** : profil en profondeur (moyenne sur les A-scans ou la choroide est
  presente) en fonction de la phase, en ecart a la moyenne du cycle -- une bande qui
  ondule verticalement = un deplacement axial au fil du cycle.
- **Intensite moyenne dans la ROI**, un trait par cycle : les cycles doivent se
  superposer si le repliement est coherent d'un bout a l'autre de l'acquisition.
- **Ecart-type entre cycles**, phase par phase : ou les cycles ne se ressemblent pas.

In [ ]:
roi = case["diag"]["roi_full_intersection"].astype(bool)
cols = np.flatnonzero(roi.any(axis=0))
csl = slice(cols.min(), cols.max() + 1) if cols.size else slice(None)

kymo = delta[:, :, csl].mean(axis=2).T  # (H, n_bins)
phases = np.arange(n_bins) / n_bins

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

k = float(np.percentile(np.abs(kymo), 99.5))
axes[0].imshow(kymo, aspect="auto", cmap="RdBu_r", vmin=-k, vmax=k,
               extent=[0, 1, kymo.shape[0], 0])
rows_roi = np.flatnonzero(roi.any(axis=1))
if rows_roi.size:
    axes[0].axhline(rows_roi.min(), color="C2", lw=0.8, ls=":")
    axes[0].axhline(rows_roi.max(), color="C2", lw=0.8, ls=":", label="etendue de la choroide")
    axes[0].legend(fontsize=7, loc="lower right")
axes[0].set_title("kymographe : profondeur x phase\n(ecart a la moyenne du cycle)", fontsize=9)
axes[0].set_xlabel("phase du cycle"); axes[0].set_ylabel("profondeur (ligne)")

for i in np.flatnonzero(case["filled"]):
    mean_roi = case["cycles"][i][:, roi].mean(axis=1)
    axes[1].plot(phases, mean_roi - mean_roi.mean(), marker="o", ms=3,
                 label=f"cycle {i}")
mean_all = cycle[:, roi].mean(axis=1)
axes[1].plot(phases, mean_all - mean_all.mean(), "k--", lw=1.6, label="moyenne")
axes[1].set_title("intensite moyenne dans la ROI", fontsize=9)
axes[1].set_xlabel("phase du cycle"); axes[1].set_ylabel("ecart (niveaux de gris)")
axes[1].legend(fontsize=7)

if case["filled"].sum() > 1:
    spread = case["cycles"][case["filled"]].std(axis=0).mean(axis=(1, 2))
    axes[2].plot(phases, spread, marker="o", ms=3, color="C3")
    axes[2].set_ylim(bottom=0)
    axes[2].set_title(f"dispersion entre les {int(case['filled'].sum())} cycles", fontsize=9)
else:
    axes[2].text(0.5, 0.5, "un seul cycle rempli", ha="center", va="center")
    axes[2].set_axis_off()
axes[2].set_xlabel("phase du cycle")

fig.tight_layout()

## 6. Le pouls et la phase qui ont servi au repliement

Le pouls repris de `conditions.csv` (combinaison optimisee des vecteurs singuliers), la
phase de Hilbert qui en est tiree, et le remplissage des bins -- un bin creux rend sa
phase bruitee dans la video.

In [ ]:
d = case["diag"]
t_u, comb = d["uniform_time"], d["combined_uniform"]
good_u = d["good_uniform"]
ts, phase_f, good_f = d["timestamps_seconds"], d["phase_per_frame"], d["good_per_frame"]

fig, axes = plt.subplots(2, 2, figsize=(13, 6))

ax = axes[0, 0]
ax.plot(t_u, comb, lw=0.9)
ax.fill_between(t_u, np.nanmin(comb), np.nanmax(comb), where=~good_u,
                color="0.85", step="mid", label="phase non fiable")
ax.set_title(f"pouls choroide entiere ({len(case['meta']['selected_indices'])} composantes SVD)",
             fontsize=9)
ax.set_xlabel("temps (s)"); ax.legend(fontsize=7)

ax = axes[0, 1]
bpm = d["freqs"] * 60.0
ax.plot(bpm, d["combined_power"], lw=1.0)
ax.fill_between(bpm, 0, d["combined_power"].max(), where=d["in_band_mask"],
                color="C1", alpha=0.15, label="bande de score")
ax.axvline(float(d["cardiac_bpm"]), color="C3", ls="--", lw=1.2,
           label=f"FC = {float(d['cardiac_bpm']):.1f} BPM")
ax.set_title("spectre du pouls (Lomb-Scargle)", fontsize=9)
ax.set_xlabel("BPM"); ax.legend(fontsize=7)

ax = axes[1, 0]
ax.plot(ts[good_f], phase_f[good_f], ".", ms=2)
ax.set_title("phase de Hilbert, par frame", fontsize=9)
ax.set_xlabel("temps (s)"); ax.set_ylabel("phase (fraction de cycle)")

# Remplissage des bins, recalcule comme le fait le repliement.
ax = axes[1, 1]
n_cycle = case["n_cycle"]
t0, span = ts[0], (ts[-1] - ts[0]) / n_cycle
bin_idx = (phase_f * n_bins).astype(int) % n_bins
width = 0.8 / n_cycle
for i in range(n_cycle):
    lo, hi = t0 + i * span, t0 + (i + 1) * span
    sel = (ts >= lo) & ((ts <= hi) if i == n_cycle - 1 else (ts < hi)) & good_f
    counts = np.bincount(bin_idx[sel], minlength=n_bins)
    ax.bar(np.arange(n_bins) + i * width - 0.4, counts, width=width,
           label=f"cycle {i} ({sel.sum()} frames)")
ax.set_title("frames par bin de phase", fontsize=9)
ax.set_xlabel("bin"); ax.set_ylabel("frames"); ax.legend(fontsize=7)

fig.suptitle(f"{case['row']['patient']} / {case['row']['condition']}", fontsize=10)
fig.tight_layout()

## 7. Ou le pouls est-il porte ?

ROI utilisee (intersection temporelle du masque) et **motif spatial** de la combinaison
retenue, remis en image a l'echelle 1 (`spatial_pattern` restreint aux traces de bloc
1x1). Rouge / bleu = les pixels qui montent / descendent ensemble dans le pouls.

In [ ]:
d = case["diag"]
mean_frame, roi_full, roi_sel = d["mean_frame"], d["roi_full_intersection"], d["roi_selected"]
pattern, scale = d["spatial_pattern"], d["scale_of_trace"]

same_roi = np.array_equal(roi_full.astype(bool), roi_sel.astype(bool))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].imshow(mean_frame)
if not same_roi:  # col_frac/row_frac ont retreci la ROI (pas le cas ici par defaut)
    ov = np.zeros((*roi_full.shape, 4)); ov[roi_full.astype(bool)] = (0.2, 0.4, 1.0, 0.25)
    axes[0].imshow(ov)
ov = np.zeros((*roi_sel.shape, 4)); ov[roi_sel.astype(bool)] = (1.0, 0.2, 0.2, 0.35)
axes[0].imshow(ov)
axes[0].set_title(
    "image moyenne + ROI des traces (rouge)" if same_roi else
    "image moyenne + masque intersecte (bleu) + ROI des traces (rouge)", fontsize=9)

if pattern.size and (scale == 1).any():
    img = np.full(roi_sel.shape, np.nan)
    img[roi_sel.astype(bool)] = pattern[scale == 1]
    v = float(np.nanpercentile(np.abs(img), 99))
    axes[1].imshow(mean_frame, alpha=0.5)
    h = axes[1].imshow(img, cmap="RdBu_r", vmin=-v, vmax=v)
    fig.colorbar(h, ax=axes[1], fraction=0.03)
    axes[1].set_title("motif spatial du pouls (echelle 1x1)", fontsize=9)
else:
    axes[1].set_axis_off()
for ax in axes:
    ax.set_xlabel("A-scan"); ax.set_ylabel("profondeur")
fig.tight_layout()

## 8. Survol de toutes les conditions

Une ligne par condition : cycle moyen a mi-phase, kymographe, et intensite ROI par
cycle. De quoi reperer d'un coup d'oeil les repliements rates (kymographe sans
structure, cycles qui ne se superposent pas) avant d'aller les regarder en detail
avec `SELECT`.

In [ ]:
MAX_CASES = 12  # None = toutes


def short_label(row):
    # "01_210713001" + "..._rigidity_OS1" -> "01 OS1"
    tail = str(row["condition"]).split("rigidity")[-1].strip("_") or str(row["condition"])[-6:]
    return f"{str(row['patient']).split('_')[0]} {tail}"


subset = cases if MAX_CASES is None else cases.head(MAX_CASES)
fig, axes = plt.subplots(len(subset), 3, figsize=(11, 2.1 * len(subset)), squeeze=False)
for i, (_, row) in enumerate(subset.iterrows()):
    try:
        cc = load_case(row)
    except Exception as exc:  # noqa: BLE001
        axes[i, 0].text(0.5, 0.5, f"illisible : {exc}", ha="center", va="center", fontsize=7)
        for ax in axes[i]:
            ax.set_axis_off()
        continue
    cyc = cc["cycles"][cc["filled"]].mean(axis=0)
    dl = cyc - cyc.mean(axis=0, keepdims=True)
    roi_i = cc["diag"]["roi_full_intersection"].astype(bool)
    ci = np.flatnonzero(roi_i.any(axis=0))
    sl = slice(ci.min(), ci.max() + 1) if ci.size else slice(None)
    ph = np.arange(cc["n_bins"]) / cc["n_bins"]

    axes[i, 0].imshow(cyc[cc["n_bins"] // 2], vmin=0, vmax=255)
    axes[i, 0].set_ylabel(short_label(row), fontsize=7)
    ky = dl[:, :, sl].mean(axis=2).T
    k = float(np.percentile(np.abs(ky), 99.5)) or 1.0
    axes[i, 1].imshow(ky, aspect="auto", cmap="RdBu_r", vmin=-k, vmax=k)
    for j in np.flatnonzero(cc["filled"]):
        y = cc["cycles"][j][:, roi_i].mean(axis=1)
        axes[i, 2].plot(ph, y - y.mean(), lw=1)
    flag = "" if row.get("rebuild_ok", True) in (True, "True", np.nan) else "  [pouls non conforme]"
    axes[i, 2].set_title(f"{row['cardiac_bpm']:.0f} BPM · "
                         f"{int(cc['filled'].sum())}/{cc['n_cycle']} cycles{flag}",
                         fontsize=7, color="C3" if flag else "black")
    for ax in axes[i, :2]:
        ax.set_xticks([]); ax.set_yticks([])
    axes[i, 2].tick_params(labelsize=6)
axes[0, 0].set_title("cycle moyen (mi-phase)", fontsize=8)
axes[0, 1].set_title("kymographe", fontsize=8)
fig.tight_layout()